# IndiVoice-DeepASR: Live Testing and Interactive Inference
### Fine-tuned Whisper Speech Recognition for Indian Accents

This notebook allows you to test the fine-tuned **IndiVoice-DeepASR** model (Whisper fine-tuned with LoRA) interactively in the Kaggle environment.

![Inference Status](https://img.shields.io/badge/Inference-Interactive-orange)
![Acoustic Profile](https://img.shields.io/badge/Acoustic-Visualization-blue)

#### Features:
1. **Zero-Touch Environment Setup**: Installs all required libraries and updates codebase.
2. **LoRA Adapter Loading**: Seamlessly merges your trained LoRA adapter checkpoints with the base Whisper model. Works with both final adapters and intermediate checkpoints.
3. **Advanced Visualizations**:
   - **Raw Waveform**: View the sound wave amplitude over time to see speaking dynamics.
   - **Mel-Spectrogram**: Visualizes the acoustic features (80 frequency bins) exactly as Whisper hears them.
4. **Random Test Manifest Profiler**: Automatically samples files from your manifest (`svarah_manifest.json`), runs inference, and computes instant WER (Word Error Rate) and CER (Character Error Rate) against ground truth transcripts.
5. **Live Gradio Web UI**: Record audio directly from your microphone or upload audio files to test transcription in real-time.

---

## 1. Zero-Touch Environment Setup
This cell sets up our working directory, updates the repository if needed, and installs the required speech processing and deployment dependencies (Gradio, PEFT, Transformers, Librosa, Jiwer).

In [ ]:
# 1. Setup working directory & clone if needed
import os
import sys

print("[LOG] Setting up environment...")
if not os.path.exists("IndiVoice-DeepASR"):
    if os.path.basename(os.getcwd()) != "IndiVoice-DeepASR":
        !git clone https://github.com/purvanshjoshi/IndiVoice-DeepASR.git
        %cd IndiVoice-DeepASR
else:
    if os.path.basename(os.getcwd()) != "IndiVoice-DeepASR":
        %cd IndiVoice-DeepASR

# Install dependencies (Gradio, PEFT, Transformers, Librosa, Jiwer, Torchaudio, BitsAndBytes)
!pip install -r requirements.txt --quiet
!pip install bitsandbytes gradio librosa matplotlib soundfile -q

# Add src to python path for imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print("[SUCCESS] Environment Setup Complete. Ready to load models.")

## 2. Load Trained Model
This cell searches for checkpoints under standard paths (such as `/kaggle/working/models/whisper-indian-lora/final_lora` or training checkpoints). If it finds a trained LoRA adapter, it merges it with the base Whisper model. If no checkpoint exists yet, it gracefully falls back to the raw pre-trained `openai/whisper-v3-small` base model, allowing you to test the inference pipeline before training completes.

In [ ]:
import os
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel, PeftConfig

def load_inference_model(model_path=None):
    """
    Loads base Whisper model along with LoRA adapters from a checkpoint.
    If no checkpoint exists, falls back to the base model for dry-run testing.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    # Determine model checkpoint path
    possible_paths = [
        model_path,
        "/kaggle/working/models/whisper-indian-lora/final_lora",
        "models/whisper-indian-final",
        "/kaggle/working/models/whisper-indian-lora"
    ]
    
    selected_path = None
    for p in possible_paths:
        if p and os.path.exists(p):
            if os.path.exists(os.path.join(p, "adapter_config.json")):
                selected_path = p
                break
            else:
                # Try finding subdirectories (e.g., checkpoints)
                subdirs = [os.path.join(p, d) for d in os.listdir(p) if d.startswith("checkpoint-")]
                if subdirs:
                    selected_path = sorted(subdirs, key=lambda x: int(x.split("-")[-1]))[-1]
                    break
                
    if selected_path:
        print(f"[INFO] Found trained checkpoint at: {selected_path}")
        peft_config = PeftConfig.from_pretrained(selected_path)
        base_model_name = peft_config.base_model_name_or_path
        print(f"Loading base model: {base_model_name}...")
        
        # Load base model (in 8-bit if CUDA is available for VRAM safety)
        if device == "cuda":
            base_model = WhisperForConditionalGeneration.from_pretrained(
                base_model_name, load_in_8bit=True, device_map="auto"
            )
        else:
            base_model = WhisperForConditionalGeneration.from_pretrained(base_model_name)
            
        print("Merging LoRA adapters...")
        model = PeftModel.from_pretrained(base_model, selected_path)
        
        # Load processor
        try:
            processor = WhisperProcessor.from_pretrained(selected_path)
        except Exception:
            processor = WhisperProcessor.from_pretrained(base_model_name)
    else:
        # Dry-run fallback
        base_model_name = "openai/whisper-v3-small"  # Lightweight model for test validation
        print(f"[WARNING] No trained LoRA adapter found in default folders.")
        print(f"Loading raw base model '{base_model_name}' for validation/testing...")
        
        if device == "cuda":
            model = WhisperForConditionalGeneration.from_pretrained(base_model_name, load_in_8bit=True, device_map="auto")
        else:
            model = WhisperForConditionalGeneration.from_pretrained(base_model_name).to(device)
            
        processor = WhisperProcessor.from_pretrained(base_model_name)
        
    model.eval()
    return model, processor

# Load the model
model, processor = load_inference_model()

## 3. Audio Preprocessing, Inference, and Visualizations
This cell defines the core visualization and transcription function. For any given audio file, it:
1. Loads the audio and resamples it to 16,000 Hz.
2. Standardizes it to mono.
3. Generates the transcription text using our model.
4. Visualizes both the **Raw Audio Waveform** and the **Mel-Spectrogram** using `librosa` and `matplotlib`.
5. Embeds an interactive HTML audio player so you can listen to it.

In [ ]:
import torchaudio
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

def visualize_and_transcribe(audio_path, model, processor, target_sr=16000):
    """
    Transcribes an audio file and plots both its Raw Waveform and Mel-Spectrogram.
    Also displays an interactive audio player.
    """
    if not os.path.exists(audio_path):
        print(f"[ERROR] Audio path {audio_path} does not exist.")
        return None
        
    # 1. Load and resample audio
    waveform, sr = torchaudio.load(audio_path)
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        waveform = resampler(waveform)
        sr = target_sr
        
    # Standardize to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    audio_np = waveform.squeeze().numpy()
    
    # 2. Run Inference
    input_features = processor(audio_np, sampling_rate=sr, return_tensors="pt").input_features.to(model.device)
    
    with torch.no_grad():
        predicted_ids = model.generate(input_features)
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
    # 3. Create Plots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Waveform Plot
    time_axis = np.linspace(0, len(audio_np) / sr, num=len(audio_np))
    ax1.plot(time_axis, audio_np, color="#2c3e50", alpha=0.85)
    ax1.set_title("Acoustic Signature: Raw Waveform", fontsize=12, fontweight='bold', pad=10)
    ax1.set_xlabel("Time (seconds)", fontsize=10)
    ax1.set_ylabel("Amplitude", fontsize=10)
    ax1.grid(True, linestyle="--", alpha=0.5)
    
    # Mel-Spectrogram Plot (Whisper 80-bin style)
    S = librosa.feature.melspectrogram(y=audio_np, sr=sr, n_mels=80, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, x_axis='time', y_axis='mel', sr=sr, fmax=8000, ax=ax2, cmap='viridis')
    fig.colorbar(img, ax=ax2, format='%+2.0f dB')
    ax2.set_title("Acoustic Fingerprint: Mel-Spectrogram (Whisper Input Feature)", fontsize=12, fontweight='bold', pad=10)
    ax2.set_xlabel("Time (seconds)", fontsize=10)
    ax2.set_ylabel("Mel Frequency (Hz)", fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Display Audio Player
    display(Audio(audio_np, rate=sr))
    
    return transcription

## 4. Benchmark Validation on Test Dataset Manifest
This cell loads the processed evaluation dataset (`data/processed/svarah_manifest.json` if available), selects a random speech sample, runs inference, and computes the Word Error Rate (WER) and Character Error Rate (CER) side-by-side against the ground truth reference transcription.

In [ ]:
import json
import random
import jiwer

manifest_path = "data/processed/svarah_manifest.json"
if os.path.exists(manifest_path):
    print(f"[INFO] Manifest loaded successfully from {manifest_path}")
    
    # Load manifest data
    manifest = []
    with open(manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            manifest.append(json.loads(line))
            
    # Sample a random speech instance
    sample = random.choice(manifest)
    sample_path = sample["audio_filepath"]
    ref_text = sample["text"]
    
    print("=" * 60)
    print("                 RANDOM TEST BENCHMARK SAMPLE            ")
    print("=" * 60)
    print(f"Audio Path: {sample_path}")
    print(f"Ground Truth Reference: \"{ref_text}\"\n")
    
    # Predict
    hyp = visualize_and_transcribe(sample_path, model, processor)
    
    if hyp:
        print(f"\nModel Hypothesis: \"{hyp}\"")
        wer_val = jiwer.wer(ref_text, hyp) * 100
        cer_val = jiwer.cer(ref_text, hyp) * 100
        print(f"Word Error Rate (WER): {wer_val:.2f}%")
        print(f"Character Error Rate (CER): {cer_val:.2f}%")
else:
    print("[WARNING] Manifest file 'data/processed/svarah_manifest.json' not found.")
    print("Ensure preprocessing is complete or link your custom dataset to run this test.")

## 5. Live Gradio Web UI Demo
This cell launches a premium Gradio Web Interface directly within the Kaggle notebook. 
- **Record Audio**: Speak into your microphone and transcribe instantly.
- **Upload File**: Upload any local WAV, MP3, or M4A file.
- **Visual Profiler**: Renders the generated Mel-Spectrogram side-by-side with the transcript.
- **Shareable Public Link**: Setting `share=True` creates a public Gradio URL, allowing you to test the speech recognition model on your mobile phone or external browser!

In [ ]:
import gradio as gr
import tempfile

def transcribe_and_visualize_gradio(audio_path):
    if audio_path is None:
        return "Please upload or record audio.", None
        
    # Load and resample audio
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)
        sr = 16000
    
    audio_np = waveform.squeeze().numpy()
    
    # Run Inference
    input_features = processor(audio_np, sampling_rate=16000, return_tensors="pt").input_features.to(model.device)
    with torch.no_grad():
        predicted_ids = model.generate(input_features)
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
    # Save Mel-Spectrogram to file for Gradio output
    fig, ax = plt.subplots(figsize=(10, 4))
    S = librosa.feature.melspectrogram(y=audio_np, sr=sr, n_mels=80, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, x_axis='time', y_axis='mel', sr=sr, fmax=8000, ax=ax, cmap='magma')
    ax.set_title("Acoustic Fingerprint (Mel-Spectrogram)")
    plt.tight_layout()
    
    temp_spec = tempfile.NamedTemporaryFile(suffix=".png", delete=False).name
    plt.savefig(temp_spec)
    plt.close(fig)
    
    return transcription, temp_spec

# Define interface with premium styling and theme
demo = gr.Interface(
    fn=transcribe_and_visualize_gradio,
    inputs=gr.Audio(type="filepath", label="Record Speech / Upload Audio File"),
    outputs=[
        gr.Textbox(label="IndiVoice Transcription", show_copy_button=True),
        gr.Image(label="Mel-Spectrogram Acoustic Profile")
    ],
    title="🎙️ IndiVoice-DeepASR: Accent Intelligent Speech Recognition",
    description="Fine-tuned Whisper model adapted for diverse Indian accents. Speak into the microphone or upload an audio file to see the transcript and the corresponding acoustic fingerprint.",
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate")
)

# Launch with sharing enabled
demo.launch(share=True)